# Project 13: Hotel Booking & Occupancy Analysis
### Industry: Hospitality & Tourism Analytics

**Business Scenario:**  
Analyze room reservations, stay durations, booking channels, and cancellations to help hotel management optimize room pricing, marketing channel mix, and front-desk staffing.

**Deliverables:**
1. Data Cleaning & Imputation Audit
2. Room-Type Revenue Summary Table
3. Booking Channel Performance & Cancellation Table
4. Monthly Occupancy & Revenue Trend
5. 5 Operational & Revenue Recommendations

## 1. Setup & Load Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set chart styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

# Load dataset
df_raw = pd.read_csv("13_hotel_bookings.csv")
print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Dataset info and statistics
df_raw.info()
df_raw.describe()

## 2. Identify Missing Values & Duplicate Booking_IDs

In [ ]:
# Audit missing values
missing = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_%': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
print("--- Missing Values ---")
print(missing[missing['Missing_Count'] > 0])

# Duplicate Booking_IDs
dup_count = df_raw['Booking_ID'].duplicated().sum()
print(f"\nDuplicate Booking_IDs: {dup_count}")

# Deduplicate keeping first occurrence
df = df_raw.drop_duplicates(subset=['Booking_ID'], keep='first').copy()
print(f"Shape after deduplication: {df.shape}")

## 3. Impute Missing Values (Justified Median Approach)

In [ ]:
# Median imputation grouped by Room_Type for Room_Rate
df['Room_Rate'] = df['Room_Rate'].fillna(df.groupby('Room_Type')['Room_Rate'].transform('median'))

# Median imputation grouped by Room_Type for Lead_Time_Days
df['Lead_Time_Days'] = df['Lead_Time_Days'].fillna(df.groupby('Room_Type')['Lead_Time_Days'].transform('median'))

# Impute missing Booking_Channel with 'Unknown'
df['Booking_Channel'] = df['Booking_Channel'].fillna('Unknown')

print("Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 4. Convert Checkin_Date & Extract Temporal Features

In [ ]:
df['Checkin_Date'] = pd.to_datetime(df['Checkin_Date'])
df['Arrival_Year_Month'] = df['Checkin_Date'].dt.strftime('%Y-%m')
df['Arrival_Month_Name'] = df['Checkin_Date'].dt.month_name()
df['Arrival_Weekday'] = df['Checkin_Date'].dt.day_name()

df[['Booking_ID', 'Checkin_Date', 'Arrival_Year_Month', 'Arrival_Month_Name', 'Arrival_Weekday']].head()

## 5. Calculate Total Bookings, Room Nights & Cancellation %

In [ ]:
total_bookings = len(df)
total_nights_demanded = df['Nights'].sum()
total_canceled = (df['Canceled'] == 'Yes').sum()
cancellation_rate = (total_canceled / total_bookings) * 100
realized_nights = df.loc[df['Canceled'] == 'No', 'Nights'].sum()

print(f"Total Bookings: {total_bookings:,}")
print(f"Total Room Nights Demanded: {total_nights_demanded:,}")
print(f"Realized Nights (Checked-in): {realized_nights:,}")
print(f"Canceled Reservations: {total_canceled:,}")
print(f"Overall Cancellation Rate: {cancellation_rate:.2f}%")

## 6. Calculate Realized Booking Revenue (Nights * Room_Rate)

In [ ]:
# Realized revenue applies only to non-canceled bookings; 0 for canceled
df['Revenue'] = np.where(df['Canceled'] == 'No', df['Nights'] * df['Room_Rate'], 0.0)
df['Potential_Revenue'] = df['Nights'] * df['Room_Rate']

total_realized_revenue = df['Revenue'].sum()
total_lost_revenue = df.loc[df['Canceled'] == 'Yes', 'Potential_Revenue'].sum()

print(f"Total Realized Revenue: INR {total_realized_revenue:,.2f}")
print(f"Revenue Lost to Cancellations: INR {total_lost_revenue:,.2f}")

## 7. Compare Room Types (Deliverable 1)

In [ ]:
room_summary = df.groupby('Room_Type').agg(
    Total_Bookings=('Booking_ID', 'count'),
    Avg_Room_Rate=('Room_Rate', 'mean'),
    Realized_Nights=('Nights', lambda s: s[df.loc[s.index, 'Canceled'] == 'No'].sum()),
    Realized_Revenue=('Revenue', 'sum'),
    Cancellation_Rate_Pct=('Canceled', lambda s: (s == 'Yes').mean() * 100)
).reset_index()

room_summary['Booking_Share_%'] = (room_summary['Total_Bookings'] / total_bookings * 100).round(2)
room_summary['Revenue_Share_%'] = (room_summary['Realized_Revenue'] / total_realized_revenue * 100).round(2)
room_summary

## 8. Rank Booking Channels (Deliverable 2)

In [ ]:
channel_summary = df.groupby('Booking_Channel').agg(
    Total_Bookings=('Booking_ID', 'count'),
    Realized_Revenue=('Revenue', 'sum'),
    Cancellation_Rate_Pct=('Canceled', lambda s: (s == 'Yes').mean() * 100)
).reset_index()

channel_summary['Booking_Share_%'] = (channel_summary['Total_Bookings'] / total_bookings * 100).round(2)
channel_summary['Revenue_Share_%'] = (channel_summary['Realized_Revenue'] / total_realized_revenue * 100).round(2)
channel_summary = channel_summary.sort_values(by='Realized_Revenue', ascending=False).reset_index(drop=True)
channel_summary['Revenue_Rank'] = channel_summary.index + 1
channel_summary

## 9. Cancellation Rates by Channel & Lead-Time Bands

In [ ]:
# Lead time categories
lead_bins = [-1, 7, 30, int(df['Lead_Time_Days'].max()) + 1]
lead_labels = ['Last-Minute (0-7d)', 'Short-Term (8-30d)', 'Advance (31+d)']
df['Lead_Time_Band'] = pd.cut(df['Lead_Time_Days'], bins=lead_bins, labels=lead_labels)

# Pivot table: Channel vs Lead-Time Band
pivot_canc = pd.crosstab(
    df['Booking_Channel'],
    df['Lead_Time_Band'],
    values=df['Canceled'] == 'Yes',
    aggfunc=lambda x: round(np.mean(x) * 100, 2)
)
pivot_canc

In [ ]:
# Multi-level Groupby: ['Room_Type', 'Booking_Channel']
multi_summary = df.groupby(['Room_Type', 'Booking_Channel']).agg(
    Total_Bookings=('Booking_ID', 'count'),
    Realized_Revenue=('Revenue', 'sum'),
    Cancellation_Rate_Pct=('Canceled', lambda s: (s == 'Yes').mean() * 100)
).round(2)
multi_summary

## 10. Top 3 Peak Revenue Months (Deliverable 3)

In [ ]:
monthly_summary = df.groupby(['Arrival_Year_Month', 'Arrival_Month_Name']).agg(
    Total_Bookings=('Booking_ID', 'count'),
    Realized_Nights=('Nights', lambda s: s[df.loc[s.index, 'Canceled'] == 'No'].sum()),
    Realized_Revenue=('Revenue', 'sum')
).reset_index().sort_values(by='Realized_Revenue', ascending=False)

top_3 = monthly_summary.head(3)
print("Top 3 Peak Revenue Months:")
top_3

## 11. Stay Duration: Leisure vs. Business Travelers

In [ ]:
traveler_summary = df.groupby('Traveler_Type').agg(
    Total_Bookings=('Booking_ID', 'count'),
    Mean_Stay_Nights=('Nights', 'mean'),
    Median_Stay_Nights=('Nights', 'median'),
    Realized_Revenue=('Revenue', 'sum'),
    Cancellation_Rate_Pct=('Canceled', lambda s: (s == 'Yes').mean() * 100)
).reset_index()

traveler_summary['Revenue_Share_%'] = (traveler_summary['Realized_Revenue'] / total_realized_revenue * 100).round(2)
traveler_summary

## 12. NumPy Statistics on Room Rates

In [ ]:
rates = df['Room_Rate'].to_numpy()
print(f"NumPy Mean:       INR {np.mean(rates):,.2f}")
print(f"NumPy Median:     INR {np.median(rates):,.2f}")
print(f"NumPy Sample Std: INR {np.std(rates, ddof=1):,.2f}")

## 13. IQR Method for Revenue Outliers

In [ ]:
realized_rev = df.loc[df['Canceled'] == 'No', 'Revenue'].to_numpy()
q1 = np.percentile(realized_rev, 25)
q3 = np.percentile(realized_rev, 75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

outliers = df[(df['Canceled'] == 'No') & (df['Revenue'] > upper_fence)]
print(f"IQR: INR {iqr:,.2f} | Upper Fence: INR {upper_fence:,.2f}")
print(f"Outlier Bookings Count: {len(outliers)}")
outliers[['Booking_ID', 'Room_Type', 'Traveler_Type', 'Nights', 'Room_Rate', 'Revenue']].head(5)

## 14. Lead-Time Bands via pd.cut()

In [ ]:
lead_perf = df.groupby('Lead_Time_Band', observed=False).agg(
    Bookings=('Booking_ID', 'count'),
    Realized_Revenue=('Revenue', 'sum'),
    Cancellation_%=('Canceled', lambda s: (s == 'Yes').mean() * 100)
).reset_index()
lead_perf

## 15. Visualizations & Executive Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Monthly Revenue & Nights
ax1 = axes[0, 0]
m_sorted = df.groupby(['Arrival_Year_Month', 'Arrival_Month_Name']).agg(
    Rev=('Revenue', 'sum'),
    Nights=('Nights', lambda s: s[df.loc[s.index, 'Canceled'] == 'No'].sum())
).reset_index().sort_values('Arrival_Year_Month')

ax1.bar([m[:3] for m in m_sorted['Arrival_Month_Name']], m_sorted['Rev'] / 1e5, color='#1f77b4', alpha=0.7)
ax1.set_ylabel('Revenue (Lakh INR)', color='#1f77b4', fontweight='bold')
ax1.set_title('Monthly Revenue & Room Nights', fontweight='bold')
ax1_twin = ax1.twinx()
ax1_twin.plot([m[:3] for m in m_sorted['Arrival_Month_Name']], m_sorted['Nights'], color='#2ca02c', marker='o', lw=2)
ax1_twin.set_ylabel('Realized Nights', color='#2ca02c', fontweight='bold')
ax1_twin.grid(False)

# 2. Cancellation by Channel
ax2 = axes[0, 1]
sns.barplot(data=channel_summary, x='Booking_Channel', y='Cancellation_Rate_Pct', ax=ax2, palette='Reds_d')
ax2.set_title('Cancellation Rate by Channel (%)', fontweight='bold')
ax2.set_ylabel('Cancellation %')

# 3. Revenue Share by Room Type
ax3 = axes[1, 0]
ax3.pie(room_summary['Realized_Revenue'], labels=room_summary['Room_Type'], autopct='%1.1f%%', colors=['#4e79a7', '#f28e2b', '#e15759'])
ax3.set_title('Revenue Share by Room Type', fontweight='bold')

# 4. Stay Duration: Leisure vs Business
ax4 = axes[1, 1]
sns.boxplot(data=df, x='Traveler_Type', y='Nights', ax=ax4, palette='Set2')
ax4.set_title('Stay Duration: Leisure vs Business', fontweight='bold')

plt.tight_layout()
plt.savefig('hotel_dashboard.png', dpi=300)
plt.show()